# 13 · RAG Evaluation

Measure a RAG system: faithfulness, answer relevance, context precision/recall, LLM-as-judge.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AfzalManihar/Agentic-AI-Learning/blob/main/13_rag_evaluation.ipynb)

**Covered in this notebook**

- See the notebook cells below

> Part of [Agentic AI Learning](https://github.com/AfzalManihar/Agentic-AI-Learning). Add your own API keys in Colab Secrets (key icon) before running.

#03-10-2026

#RAGS-(RAG Assessment - For Checking Accuracy Of Our Agent)

#Accuracy of LLM

(chunking,retrival,faithfullness,answer relivance,precision,recall)

In [ ]:
# ---------- Cell 1: Install libraries + load the Groq API key ----------

# sentence-transformers -> turns text into vectors (used for retrieval AND for answer relevance)
# langchain-groq        -> Groq chat models (the RAG generator and the judge)
# langchain-core        -> message primitives
# pandas / numpy        -> result tables and vector maths

%pip install -q sentence-transformers langchain-groq langchain-core pandas numpy

import os
from getpass import getpass


def load_api_key(secret_name: str = "groq_api3") -> str:
    """Return the Groq API key from Colab Secrets, the environment, or a prompt."""
    try:
        from google.colab import userdata
        return userdata.get(secret_name)
    except Exception:
        pass

    return os.environ.get(secret_name) or getpass(f"Enter {secret_name}: ")


# Load API key from Colab Secret: groq_api3
API_KEY = load_api_key("groq_api3")

print("API key loaded:", bool(API_KEY))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 9.1 MB/s eta 0:00:00
API key loaded: True


In [ ]:
# ---------- Cell 2: Imports, configuration, models and helpers ----------

import json
import re
import threading
import time
from dataclasses import dataclass

import numpy as np
import pandas as pd
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_groq import ChatGroq
from sentence_transformers import SentenceTransformer


# ---------- Configuration: all tunable settings in one place ----------
@dataclass
class Config:
    generator_model: str = "openai/gpt-oss-20b"    # the SMALL model that powers the RAG system
    judge_model: str = "openai/gpt-oss-120b"       # the BIGGER model that acts as judge (LLM as Judge)
    embed_model: str = "sentence-transformers/all-MiniLM-L6-v2"   # text -> vector (retrieval + relevance)
    top_k: int = 3                                  # how many chunks the retriever returns
    verbose: bool = True


CFG = Config()

# ---------- Models ----------
# The generator answers questions. The judge ONLY evaluates, and always replies in JSON.
generator_llm = ChatGroq(api_key=API_KEY, model=CFG.generator_model, temperature=0)
judge_llm = ChatGroq(
    api_key=API_KEY,
    model=CFG.judge_model,
    temperature=0,
    model_kwargs={"response_format": {"type": "json_object"}},
)

# Embedding model: runs locally (no API key). Downloaded once, about 90 MB.
embedder = SentenceTransformer(CFG.embed_model)


# ---------- Call counter: how many calls does the generator make vs. the judge? ----------
class CallCounter:
    """Thread-safe counter of LLM calls per role ('generator' / 'judge')."""

    def __init__(self) -> None:
        self._lock = threading.Lock()
        self.counts = {"generator": 0, "judge": 0}

    def add(self, role: str) -> None:
        with self._lock:
            self.counts[role] += 1

    def reset(self) -> None:
        with self._lock:
            self.counts = {"generator": 0, "judge": 0}


COUNTER = CallCounter()


# ---------- LLM calls with automatic rate-limit retry ----------
def _wait_seconds(message: str, attempt: int) -> float:
    """Read 'try again in 150ms / 2.5s' from Groq's error; otherwise back off exponentially."""
    match = re.search(r"try again in ([0-9.]+)(ms|s)", message)
    if match:
        value = float(match.group(1))
        seconds = value / 1000 if match.group(2) == "ms" else value
        return min(seconds + 1.0, 60.0)             # +1s safety margin
    return min(2.0 ** attempt, 30.0)


def call_llm(model, messages, role: str, max_retries: int = 8):
    """Single entry point for every LLM call: counted once, and retried on Groq rate limits."""
    COUNTER.add(role)
    for attempt in range(max_retries):
        try:
            return model.invoke(messages)
        except Exception as exc:
            text = str(exc)
            is_rate_limit = type(exc).__name__ == "RateLimitError" or "rate_limit" in text or "429" in text
            if not is_rate_limit or attempt == max_retries - 1:
                raise                               # other errors (auth, bad request...) are not hidden
            wait = _wait_seconds(text, attempt)
            if CFG.verbose:
                print(f"⏳ Rate limit reached, waiting {wait:.1f}s (retry {attempt + 1}/{max_retries - 1})...")
            time.sleep(wait)


def ask_json(system: str, user: str, retries: int = 3) -> dict:
    """Ask the JUDGE for a JSON answer and return it as a dict. Retries on empty/invalid JSON."""
    last_error = None
    for attempt in range(retries):
        nudge = "" if attempt == 0 else "\n\n(Reply with ONE valid JSON object only.)"
        try:
            reply = call_llm(judge_llm, [SystemMessage(system), HumanMessage(user + nudge)], "judge")
            return json.loads(reply.content)
        except Exception as exc:
            retryable = isinstance(exc, json.JSONDecodeError) or "json_validate_failed" in str(exc)
            if not retryable:
                raise
            last_error = exc
            time.sleep(1.5)
    raise RuntimeError(f"Judge returned no valid JSON after {retries} attempts.") from last_error


# ---------- Embedding helpers ----------
def embed(texts: list) -> np.ndarray:
    """Turn texts into unit-length vectors, so cosine similarity is just a dot product."""
    return embedder.encode(texts, normalize_embeddings=True)


def cosine(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.dot(a, b))


# ---------- Smoke tests ----------
print("Generator:", call_llm(
    generator_llm, [HumanMessage("Reply with exactly three words: generator is working")], "generator"
).content)
print("Judge    :", ask_json("Reply ONLY with JSON.", 'Return {"status": "judge is working"}'))

vecs = embed(["How long is the warranty?", "What is the warranty period?", "How do I reset my password?"])
print("Embedding shape:", vecs.shape)
print("Similar questions   :", round(cosine(vecs[0], vecs[1]), 2))
print("Unrelated questions :", round(cosine(vecs[0], vecs[2]), 2))

COUNTER.reset()   # do not count the smoke tests

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Generator: generator is working
Judge    : {'status': 'judge is working'}
Embedding shape: (3, 384)
Similar questions   : 0.93
Unrelated questions : 0.2


In [ ]:
# ---------- Cell 3: Documents (the knowledge base of our RAG system) ----------

# A fictional Help Center for "Zephyr" electric scooters. All facts are made up for this demo.
# Every document has several short paragraphs, each holding a few related facts.
# Why: Cell 4 will cut these texts into chunks, and chunk boundaries decide what retrieval can find.
#
# Note: there is NO document about EMI/financing. Later we will ask about it on purpose,
# to see whether the RAG system admits "I don't know" or makes something up (a hallucination).

DOCUMENTS = [
    {
        "doc_id": "warranty",
        "title": "Warranty Policy",
        "text": (
            "Zephyr S1 comes with a 3-year warranty. Zephyr S2 Plus comes with a 2-year warranty.\n\n"
            "The battery has its own cover: 2 years on the S1 and 1 year on the S2 Plus, or until the battery "
            "falls below 70 percent of its original capacity, whichever comes first.\n\n"
            "The warranty does not cover accident damage, water immersion or repairs done by unauthorised workshops.\n\n"
            "To claim warranty, contact support with your invoice and the scooter's serial number. "
            "Claims are reviewed within 5 working days."
        ),
    },
    {
        "doc_id": "charging",
        "title": "Charging and Battery Care",
        "text": (
            "With the standard charger, the Zephyr S1 charges from 0 to 100 percent in 6 hours. "
            "The Zephyr S2 Plus supports fast charging and reaches 80 percent in 90 minutes.\n\n"
            "Always charge in a dry place at temperatures between 5 and 40 degrees Celsius.\n\n"
            "For long storage, keep the battery at about 50 percent charge and top it up every 2 months.\n\n"
            "Do not use third-party chargers. They void the battery warranty."
        ),
    },
    {
        "doc_id": "returns",
        "title": "Returns and Refunds",
        "text": (
            "You can return a new scooter within 15 days of delivery, provided it has been ridden less than 100 km.\n\n"
            "Refunds go to the original payment method within 7 working days after the scooter passes inspection.\n\n"
            "A pickup fee of 1500 INR is deducted from the refund, unless the scooter arrived damaged.\n\n"
            "Unused accessories such as helmets can be returned within 30 days."
        ),
    },
    {
        "doc_id": "service",
        "title": "Service and Maintenance",
        "text": (
            "The first service is free and is due at 500 km or 3 months, whichever comes first. "
            "After that, service is due every 3000 km or 6 months.\n\n"
            "A basic service costs 800 INR. Replacing the brake pads costs 600 INR per pair.\n\n"
            "Authorised service centres in Pune are in Kothrud, Hinjewadi and Viman Nagar.\n\n"
            "Check the tyre pressure every two weeks: 28 psi for the front tyre and 32 psi for the rear tyre."
        ),
    },
    {
        "doc_id": "delivery",
        "title": "Delivery and Registration",
        "text": (
            "Delivery takes 7 to 10 working days after payment is confirmed.\n\n"
            "The delivery address cannot be changed after the scooter has been dispatched.\n\n"
            "Zephyr handles registration and the number plate for a fee of 2500 INR.\n\n"
            "A one-year insurance policy is included in the on-road price."
        ),
    },
]

# ---------- Overview of the knowledge base ----------
overview = pd.DataFrame([
    {
        "doc_id": d["doc_id"],
        "title": d["title"],
        "paragraphs": len(d["text"].split("\n\n")),
        "words": len(d["text"].split()),
    }
    for d in DOCUMENTS
])
display(overview)
print(f"Total: {len(DOCUMENTS)} documents, {overview['words'].sum()} words")

# ---------- Preview one document ----------
print("\n--- Preview: warranty ---")
print(DOCUMENTS[0]["text"])

,doc_id,title,paragraphs,words
0,warranty,Warranty Policy,4,83
1,charging,Charging and Battery Care,4,72
2,returns,Returns and Refunds,4,63
3,service,Service and Maintenance,4,75
4,delivery,Delivery and Registration,4,46


Total: 5 documents, 339 words

--- Preview: warranty ---
Zephyr S1 comes with a 3-year warranty. Zephyr S2 Plus comes with a 2-year warranty.

The battery has its own cover: 2 years on the S1 and 1 year on the S2 Plus, or until the battery falls below 70 percent of its original capacity, whichever comes first.

The warranty does not cover accident damage, water immersion or repairs done by unauthorised workshops.

To claim warranty, contact support with your invoice and the scooter's serial number. Claims are reviewed within 5 working days.


In [ ]:
# ---------- Cell 4: Chunking (cutting documents into retrievable pieces) ----------

# Why chunk at all? A retriever does not return whole documents, it returns small pieces (chunks).
# The way we cut decides what the retriever CAN find:
#   - cut too small / in the wrong place -> one fact is split over two chunks (recall suffers)
#   - cut too big                        -> a chunk holds a lot of noise (precision suffers)
# Cell 12 measures this effect with the RAGAS metrics. Here we only build the chunkers.


def chunk_by_paragraph(doc: dict) -> list:
    """Strategy 1: one chunk per paragraph. Keeps related facts together (our docs have clean paragraphs)."""
    paragraphs = [p.strip() for p in doc["text"].split("\n\n") if p.strip()]
    return [
        {"chunk_id": f"{doc['doc_id']}-{i}", "doc_id": doc["doc_id"], "text": p}
        for i, p in enumerate(paragraphs)
    ]


def chunk_by_words(doc: dict, size: int = 25, overlap: int = 5) -> list:
    """Strategy 2: fixed-size window of `size` words, moving forward by (size - overlap) words.

    The overlap repeats a few words at each boundary, so a fact cut at the edge appears (partly) in both chunks.
    """
    if overlap >= size:
        raise ValueError("overlap must be smaller than size")
    words = doc["text"].split()
    step = size - overlap
    chunks, start = [], 0
    while start < len(words):
        chunks.append({
            "chunk_id": f"{doc['doc_id']}-{len(chunks)}",
            "doc_id": doc["doc_id"],
            "text": " ".join(words[start:start + size]),
        })
        if start + size >= len(words):      # this window already reaches the end of the document
            break
        start += step
    return chunks


def build_chunks(strategy: str) -> list:
    """Chunk ALL documents with one strategy and return a flat list of chunks."""
    chunker = {"paragraph": chunk_by_paragraph, "words25": chunk_by_words}[strategy]
    return [chunk for doc in DOCUMENTS for chunk in chunker(doc)]


# ---------- Build both versions of the chunk store ----------
CHUNK_SETS = {name: build_chunks(name) for name in ["paragraph", "words25"]}

# ---------- Compare the two strategies ----------
rows = []
for name, chunks in CHUNK_SETS.items():
    lengths = [len(c["text"].split()) for c in chunks]
    rows.append({
        "strategy": name,
        "chunks": len(chunks),
        "avg words": round(sum(lengths) / len(lengths), 1),
        "min words": min(lengths),
        "max words": max(lengths),
    })
display(pd.DataFrame(rows))

# ---------- Look at the first chunks of the warranty document under each strategy ----------
for name, chunks in CHUNK_SETS.items():
    print(f"\n--- {name}: first 3 chunks of 'warranty' ---")
    for chunk in [c for c in chunks if c["doc_id"] == "warranty"][:3]:
        print(f"[{chunk['chunk_id']}] {chunk['text']}")

,strategy,chunks,avg words,min words,max words
0,paragraph,20,16.9,10,33
1,words25,18,22.4,6,25



--- paragraph: first 3 chunks of 'warranty' ---
[warranty-0] Zephyr S1 comes with a 3-year warranty. Zephyr S2 Plus comes with a 2-year warranty.
[warranty-1] The battery has its own cover: 2 years on the S1 and 1 year on the S2 Plus, or until the battery falls below 70 percent of its original capacity, whichever comes first.
[warranty-2] The warranty does not cover accident damage, water immersion or repairs done by unauthorised workshops.

--- words25: first 3 chunks of 'warranty' ---
[warranty-0] Zephyr S1 comes with a 3-year warranty. Zephyr S2 Plus comes with a 2-year warranty. The battery has its own cover: 2 years on the
[warranty-1] cover: 2 years on the S1 and 1 year on the S2 Plus, or until the battery falls below 70 percent of its original capacity,
[warranty-2] percent of its original capacity, whichever comes first. The warranty does not cover accident damage, water immersion or repairs done by unauthorised workshops. To claim


In [ ]:
# ---------- Cell 5: Retrieval (finding the chunks that match a question) ----------

# How it works:
#   1. Every chunk is turned into a vector ONCE (this is our "vector store"; here it is just a matrix).
#   2. A question is turned into a vector too.
#   3. Cosine similarity between the question and every chunk ranks the chunks.
#   4. The top-k chunks are returned. THE RETRIEVER ALWAYS RETURNS k CHUNKS, even if none is relevant.
#      That last point is exactly why "context precision" exists.


@dataclass
class ChunkStore:
    """A set of chunks together with their vectors (computed once)."""
    name: str
    chunks: list
    vectors: np.ndarray        # shape: (number of chunks, vector size)


def build_store(name: str) -> ChunkStore:
    chunks = CHUNK_SETS[name]
    vectors = embed([c["text"] for c in chunks])
    return ChunkStore(name=name, chunks=chunks, vectors=vectors)


STORES = {name: build_store(name) for name in CHUNK_SETS}


def retrieve(question: str, store: ChunkStore, k: int = None) -> list:
    """Return the top-k chunks for a question, each with its similarity score."""
    k = k or CFG.top_k
    question_vector = embed([question])[0]
    scores = store.vectors @ question_vector              # cosine similarity (vectors are unit length)
    top_indices = np.argsort(scores)[::-1][:k]            # best first
    return [{**store.chunks[i], "score": round(float(scores[i]), 3)} for i in top_indices]


def show_retrieval(question: str, store_name: str) -> None:
    """Print the retrieved chunks for a question (text shortened to one line)."""
    print(f"\nQ: {question}   [store: {store_name}]")
    for rank, chunk in enumerate(retrieve(question, STORES[store_name]), start=1):
        preview = chunk["text"][:85] + ("..." if len(chunk["text"]) > 85 else "")
        print(f"  {rank}. score={chunk['score']:.2f}  [{chunk['chunk_id']}]  {preview}")


# ---------- Demo 1: a normal question ----------
show_retrieval("How long is the warranty on the Zephyr S1?", "paragraph")

# ---------- Demo 2: the SAME question, but with the word-window chunks ----------
show_retrieval("How long is the warranty on the Zephyr S1?", "words25")

# ---------- Demo 3: a question the knowledge base cannot answer (there is no EMI document) ----------
show_retrieval("Do you offer EMI or financing options?", "paragraph")


Q: How long is the warranty on the Zephyr S1?   [store: paragraph]
  1. score=0.81  [warranty-0]  Zephyr S1 comes with a 3-year warranty. Zephyr S2 Plus comes with a 2-year warranty.
  2. score=0.49  [warranty-1]  The battery has its own cover: 2 years on the S1 and 1 year on the S2 Plus, or until ...
  3. score=0.47  [charging-0]  With the standard charger, the Zephyr S1 charges from 0 to 100 percent in 6 hours. Th...

Q: How long is the warranty on the Zephyr S1?   [store: words25]
  1. score=0.75  [warranty-0]  Zephyr S1 comes with a 3-year warranty. Zephyr S2 Plus comes with a 2-year warranty. ...
  2. score=0.56  [warranty-1]  cover: 2 years on the S1 and 1 year on the S2 Plus, or until the battery falls below ...
  3. score=0.51  [charging-3]  2 months. Do not use third-party chargers. They void the battery warranty.

Q: Do you offer EMI or financing options?   [store: paragraph]
  1. score=0.22  [delivery-3]  A one-year insurance policy is included in the on-road price.
  2. sc

In [ ]:
# ---------- Cell 6: The RAG system (retrieve -> generate) ----------

# Two prompt styles on purpose. RAGAS exists to tell a healthy RAG system from a broken one,
# so we need something to compare:
#   "strict" : answer ONLY from the context, and admit when the context has no answer.
#   "loose"  : "be helpful" and fill the gaps with general knowledge. Risky: it may INVENT facts.
RAG_PROMPTS = {
    "strict": (
        "You are a customer support assistant for Zephyr electric scooters. "
        "Answer the question using ONLY the context below. "
        "If the context does not contain the answer, reply exactly: I don't know based on the available documents. "
        "Keep the answer to 1-3 short sentences."
    ),
    "loose": (
        "You are a helpful customer support assistant for Zephyr electric scooters. "
        "Use the context if it helps, and answer the customer as helpfully as you can. "
        "Keep the answer to 1-3 short sentences."
    ),
}


def format_context(chunks: list) -> str:
    """Number the retrieved chunks so the model sees them as a clear list."""
    return "\n".join(f"[{i}] {chunk['text']}" for i, chunk in enumerate(chunks, start=1))


def rag_answer(question: str, store_name: str = "paragraph", k: int = None, style: str = "strict") -> dict:
    """Full RAG pipeline: retrieve chunks, then let the generator answer from them.

    Returns everything RAGAS needs later: the question, the retrieved contexts and the answer.
    """
    chunks = retrieve(question, STORES[store_name], k)
    user_message = f"Context:\n{format_context(chunks)}\n\nQuestion: {question}"
    reply = call_llm(
        generator_llm,
        [SystemMessage(RAG_PROMPTS[style]), HumanMessage(user_message)],
        "generator",
    )
    return {
        "question": question,
        "store": store_name,
        "style": style,
        "chunk_ids": [c["chunk_id"] for c in chunks],
        "contexts": [c["text"] for c in chunks],
        "answer": reply.content.strip(),
    }


def show_rag(result: dict) -> None:
    print(f"\nQ ({result['style']} prompt): {result['question']}")
    print(f"   retrieved: {result['chunk_ids']}")
    print(f"   answer   : {result['answer']}")


# ---------- Demo 1: a question the documents CAN answer ----------
show_rag(rag_answer("How long is the warranty on the Zephyr S1?", style="strict"))

# ---------- Demo 2 and 3: a question the documents can NOT answer (no EMI document exists) ----------
show_rag(rag_answer("Do you offer EMI or financing options?", style="strict"))
show_rag(rag_answer("Do you offer EMI or financing options?", style="loose"))


Q (strict prompt): How long is the warranty on the Zephyr S1?
   retrieved: ['warranty-0', 'warranty-1', 'charging-0']
   answer   : The Zephyr S1 has a 3‑year warranty.

Q (strict prompt): Do you offer EMI or financing options?
   retrieved: ['delivery-3', 'delivery-2', 'service-0']
   answer   : I don't know based on the available documents.

Q (loose prompt): Do you offer EMI or financing options?
   retrieved: ['delivery-3', 'delivery-2', 'service-0']
   answer   : We don’t currently offer EMI or financing plans for Zephyr scooters. All purchases are paid in full at the time of sale. If you need more flexible payment options, let us know and we’ll see what can be arranged.


In [ ]:
# ---------- Cell 7: Faithfulness (hallucination detection) ----------

# Question answered by this metric: does the answer contain ONLY claims supported by the retrieved context?
# Exactly the two judge steps from your notes:
#   Step 1: the judge LLM breaks the answer into individual claims.
#   Step 2: the judge checks EACH claim against the retrieved chunks.
# Score = supported claims / total claims.
#   1.0       -> every claim is backed by the context
#   below 1.0 -> the LLM added information that is NOT in the retrieved documents (hallucination)

CLAIMS_PROMPT = (
    "You split an answer into atomic factual claims. "
    "Each claim must be a short, self-contained statement (replace pronouns like 'it' with the real subject). "
    "Do NOT include greetings, opinions, or refusals such as 'I don't know'. "
    'Reply ONLY with JSON: {"claims": ["claim 1", "claim 2"]}. If there are no factual claims, return {"claims": []}.'
)

VERIFY_PROMPT = (
    "You are a strict fact checker. You get a numbered list of claims and a context. "
    "A claim is SUPPORTED only if the context states it or it follows directly from the context. "
    "Use ONLY the context. Do NOT use your own knowledge, even if the claim sounds true. "
    'Reply ONLY with JSON: {"verdicts": [{"index": 1, "supported": true, "reason": "short reason"}]} '
    "with exactly one verdict per claim."
)


def extract_claims(question: str, answer: str) -> list:
    """Judge step 1: split the answer into individual claims."""
    out = ask_json(CLAIMS_PROMPT, f"Question: {question}\n\nAnswer: {answer}")
    return [c for c in out.get("claims", []) if isinstance(c, str) and c.strip()]


def verify_claims(claims: list, contexts: list) -> list:
    """Judge step 2: for each claim, is it supported by the retrieved context? Returns a list of booleans."""
    numbered = "\n".join(f"{i}. {claim}" for i, claim in enumerate(claims, start=1))
    context_text = "\n".join(f"- {c}" for c in contexts)
    out = ask_json(VERIFY_PROMPT, f"Context:\n{context_text}\n\nClaims:\n{numbered}")
    supported = {}
    for verdict in out.get("verdicts", []):
        try:
            supported[int(verdict["index"])] = (bool(verdict["supported"]), verdict.get("reason", ""))
        except (KeyError, ValueError, TypeError):
            continue
    # A claim the judge forgot to rule on counts as NOT supported (the safe choice).
    return [supported.get(i, (False, "no verdict returned")) for i in range(1, len(claims) + 1)]


def faithfulness(question: str, answer: str, contexts: list) -> dict:
    """Faithfulness score between 0 and 1, plus the claims and the judge's verdicts."""
    claims = extract_claims(question, answer)
    if not claims:
        # Nothing factual was claimed (for example an honest "I don't know"), so nothing can be unsupported.
        return {"score": 1.0, "claims": [], "verdicts": []}
    verdicts = verify_claims(claims, contexts)
    score = sum(1 for ok, _ in verdicts if ok) / len(claims)
    return {"score": round(score, 2), "claims": claims, "verdicts": verdicts}


def show_faithfulness(label: str, question: str, answer: str, result: dict) -> None:
    print(f"\n=== {label} ===")
    print(f"Q: {question}")
    print(f"A: {answer}")
    if not result["claims"]:
        print("   (no factual claims in the answer)")
    for claim, (ok, reason) in zip(result["claims"], result["verdicts"]):
        print(f"   {'✅' if ok else '❌'} {claim}   <- {reason}")
    print(f"   FAITHFULNESS = {result['score']}")


# ---------- Demo A: a hand-written answer with ONE invented claim (guaranteed hallucination) ----------
question_a = "How long is the warranty on the Zephyr S1?"
context_a = [DOCUMENTS[0]["text"].split("\n\n")[0]]          # only the first warranty paragraph
answer_a = "The Zephyr S1 has a 3-year warranty and it also includes free roadside assistance for life."
show_faithfulness("Demo A: invented claim", question_a, answer_a, faithfulness(question_a, answer_a, context_a))

# ---------- Demos B, C, D: answers from our real RAG system (Cell 6) ----------
demo_runs = [
    ("Demo B: strict prompt, answerable question", "How long is the warranty on the Zephyr S1?", "strict"),
    ("Demo C: strict prompt, NOT answerable (EMI)", "Do you offer EMI or financing options?", "strict"),
    ("Demo D: loose prompt, NOT answerable (EMI)", "Do you offer EMI or financing options?", "loose"),
]
for label, question, style in demo_runs:
    rag = rag_answer(question, style=style)
    result = faithfulness(rag["question"], rag["answer"], rag["contexts"])
    show_faithfulness(label, rag["question"], rag["answer"], result)


=== Demo A: invented claim ===
Q: How long is the warranty on the Zephyr S1?
A: The Zephyr S1 has a 3-year warranty and it also includes free roadside assistance for life.
   ✅ The Zephyr S1 has a 3-year warranty.   <- Context states Zephyr S1 comes with a 3-year warranty.
   ❌ The Zephyr S1 includes free roadside assistance for life.   <- Context does not mention free roadside assistance for life.
   FAITHFULNESS = 0.5

=== Demo B: strict prompt, answerable question ===
Q: How long is the warranty on the Zephyr S1?
A: The Zephyr S1 has a 3‑year warranty.
   ✅ The Zephyr S1 has a 3-year warranty.   <- Context explicitly states Zephyr S1 has a 3-year warranty
   FAITHFULNESS = 1.0

=== Demo C: strict prompt, NOT answerable (EMI) ===
Q: Do you offer EMI or financing options?
A: I don't know based on the available documents.
   ❌ The answerer does not know whether EMI or financing options are offered based on the available documents.   <- Context contains no information about EMI, financ

In [ ]:
# ---------- Cell 7b: Fix for Demo C (an honest "I don't know" must NOT be punished) ----------

# Problem seen in your output: for the answer "I don't know based on the available documents." the claim
# extractor invented a claim ABOUT THE ANSWERER ("the answerer does not know whether..."), the fact checker
# rightly said "not in the context", and an honest refusal scored FAITHFULNESS = 0.0.
# That is wrong: a refusal states no fact about the world, so nothing in it can be unsupported.
# It also made the honest strict answer look exactly as bad as the hallucinating loose answer (both 0.0).
#
# Fix, in two layers (these definitions REPLACE the ones from Cell 7):
#   1. An exact refusal (the sentence our strict RAG prompt asks for) is recognised in code, with no judge call.
#   2. For any other wording, the judge must flag `refusal: true` and return no claims.

CLAIMS_PROMPT = (
    "You split an answer into atomic factual claims about the WORLD (products, policies, numbers, rules). "
    "Each claim must be a short, self-contained statement (replace pronouns like 'it' with the real subject). "
    "NEVER write a claim about the answerer itself (for example 'the answerer does not know'). "
    "Do NOT include greetings or opinions. "
    "If the answer only says that it does not know or cannot answer, set refusal to true and claims to []. "
    'Reply ONLY with JSON: {"refusal": false, "claims": ["claim 1", "claim 2"]}.'
)

STRICT_REFUSAL = "i don't know based on the available documents"


def extract_claims(question: str, answer: str) -> list:
    """Judge step 1: split the answer into claims. A refusal returns no claims."""
    normalized = answer.lower().replace("’", "'").strip().rstrip(".")
    if normalized == STRICT_REFUSAL:                  # layer 1: exact refusal, no judge call needed
        return []
    out = ask_json(CLAIMS_PROMPT, f"Question: {question}\n\nAnswer: {answer}")
    if out.get("refusal") is True:                    # layer 2: the judge recognised a refusal
        return []
    return [c for c in out.get("claims", []) if isinstance(c, str) and c.strip()]


# ---------- Re-run the demos: A (invented claim), C (honest refusal), D (hallucinated policy) ----------
show_faithfulness("Demo A: invented claim", question_a, answer_a, faithfulness(question_a, answer_a, context_a))

for label, question, style in demo_runs[1:]:        # Demo C and Demo D from Cell 7
    rag = rag_answer(question, style=style)
    result = faithfulness(rag["question"], rag["answer"], rag["contexts"])
    show_faithfulness(label, rag["question"], rag["answer"], result)


=== Demo A: invented claim ===
Q: How long is the warranty on the Zephyr S1?
A: The Zephyr S1 has a 3-year warranty and it also includes free roadside assistance for life.
   ✅ The Zephyr S1 has a 3-year warranty.   <- Context states Zephyr S1 comes with a 3-year warranty.
   ❌ The Zephyr S1 includes free roadside assistance for life.   <- Context does not mention free roadside assistance.
   FAITHFULNESS = 0.5

=== Demo C: strict prompt, NOT answerable (EMI) ===
Q: Do you offer EMI or financing options?
A: I don't know based on the available documents.
   (no factual claims in the answer)
   FAITHFULNESS = 1.0

=== Demo D: loose prompt, NOT answerable (EMI) ===
Q: Do you offer EMI or financing options?
A: We don’t currently offer EMI or financing plans for Zephyr scooters. All purchases are paid in full at the time of sale. If you need more flexible payment options, let us know and we’ll see what can be arranged.
   ❌ Zephyr scooters are not currently offered with EMI or financing pl

In [ ]:
# ---------- Cell 8: Answer Relevance (did the answer address the question that was asked?) ----------

# Exactly the idea from your notes:
#   Step 1: the judge LLM reads ONLY the answer and writes several hypothetical questions
#           that this answer would be a good response to.
#   Step 2: we measure how similar those questions are to the ORIGINAL question (embedding cosine similarity).
# High similarity -> the answer is on-topic. Low similarity -> the answer drifted or addressed a different question.
# This catches what Faithfulness cannot: a fluent, well-grounded answer to a slightly different question.


def generate_questions(answer: str, n: int = 3) -> list:
    """Judge step 1: from the answer alone, write n questions it would answer well."""
    system = (
        f"You are given an ANSWER from a support assistant. Write {n} different questions that this answer "
        "would be a good, direct response to. Base the questions ONLY on the answer text, "
        "never on outside knowledge. "
        'Reply ONLY with JSON: {"questions": ["question 1", "question 2", "question 3"]}.'
    )
    out = ask_json(system, f"ANSWER: {answer}")
    return [q for q in out.get("questions", []) if isinstance(q, str) and q.strip()][:n]


def answer_relevance(question: str, answer: str, n: int = 3) -> dict:
    """Answer relevance = average similarity between the original question and the generated questions."""
    generated = generate_questions(answer, n)
    if not generated:
        return {"score": 0.0, "generated": [], "similarities": []}
    vectors = embed([question] + generated)                       # row 0 = original question
    similarities = [cosine(vectors[0], v) for v in vectors[1:]]
    score = max(0.0, sum(similarities) / len(similarities))      # cosine can be negative; clip at 0
    return {"score": round(score, 2), "generated": generated, "similarities": [round(s, 2) for s in similarities]}


def show_relevance(label: str, question: str, answer: str, result: dict) -> None:
    print(f"\n=== {label} ===")
    print(f"Q: {question}")
    print(f"A: {answer}")
    for q, sim in zip(result["generated"], result["similarities"]):
        print(f"   sim={sim:.2f}  generated question: {q}")
    print(f"   ANSWER RELEVANCE = {result['score']}")


question = "How long is the warranty on the Zephyr S1?"

# ---------- Demo A: an on-topic answer ----------
answer_on_topic = "The Zephyr S1 comes with a 3-year warranty."
show_relevance("Demo A: on-topic answer", question, answer_on_topic, answer_relevance(question, answer_on_topic))

# ---------- Demo B: fluent and GROUNDED, but it answers a DIFFERENT question (about charging) ----------
answer_drifted = "The Zephyr S1 charges from 0 to 100 percent in 6 hours with the standard charger."
show_relevance("Demo B: grounded but off-topic", question, answer_drifted, answer_relevance(question, answer_drifted))

# Is Demo B faithful? Yes: every claim IS in the charging chunk. Faithfulness cannot see the drift.
charging_context = [DOCUMENTS[1]["text"].split("\n\n")[0]]
faith_b = faithfulness(question, answer_drifted, charging_context)
print(f"   (for comparison) FAITHFULNESS of Demo B = {faith_b['score']}  <- fully grounded, still the wrong answer")

# ---------- Demo C: an honest refusal ----------
question_emi = "Do you offer EMI or financing options?"
answer_refusal = "I don't know based on the available documents."
show_relevance("Demo C: honest refusal", question_emi, answer_refusal, answer_relevance(question_emi, answer_refusal))


=== Demo A: on-topic answer ===
Q: How long is the warranty on the Zephyr S1?
A: The Zephyr S1 comes with a 3-year warranty.
   sim=0.97  generated question: What is the warranty length for the Zephyr S1?
   sim=0.93  generated question: How many years does the Zephyr S1 warranty cover?
   sim=0.96  generated question: Is there a warranty on the Zephyr S1, and what duration does it have?
   ANSWER RELEVANCE = 0.95

=== Demo B: grounded but off-topic ===
Q: How long is the warranty on the Zephyr S1?
A: The Zephyr S1 charges from 0 to 100 percent in 6 hours with the standard charger.
   sim=0.55  generated question: How long does it take to fully charge the Zephyr S1 with the standard charger?
   sim=0.48  generated question: What is the charging time from 0% to 100% for the Zephyr S1 using the included charger?
   sim=0.56  generated question: How many hours does the Zephyr S1 need to reach a full charge with the standard charger?
   ANSWER RELEVANCE = 0.53
   (for comparison) FAITHFUL

In [ ]:
# ---------- Cell 9: Context Precision and Context Recall (judging the RETRIEVER) ----------

# The two retrieval metrics from your notes:
#   Context Precision: out of the chunks retrieved, how many were relevant?      (are we returning noise?)
#   Context Recall   : out of all the relevant information, how much was retrieved? (did we miss something?)
#
# To measure them we need GROUND TRUTH: for every test question, what information is actually needed.
# We store it as short "evidence" snippets (5-7 words each) that must appear in the retrieved context.
#   - a chunk is RELEVANT if it contains at least one evidence snippet  -> used for precision
#   - recall = how many of the evidence snippets appear in the retrieved chunks
#     (counted per snippet, not per chunk, so overlapping chunks cannot count the same fact twice)

# ---------- The test set: questions with pre-written correct answers (also used in Cells 10-12) ----------
EVAL_SET = [
    {
        "id": "q1",
        "question": "How long is the warranty on the Zephyr S1?",
        "ground_truth": "The Zephyr S1 has a 3-year warranty.",
        "evidence": ["comes with a 3-year warranty"],
    },
    {
        "id": "q2",
        "question": "How long does the Zephyr S2 Plus take to reach 80 percent charge?",
        "ground_truth": "The Zephyr S2 Plus reaches 80 percent charge in 90 minutes with fast charging.",
        "evidence": ["reaches 80 percent in 90 minutes"],
    },
    {
        "id": "q3",
        "question": "How many days do I have to return a new scooter, and how much is the pickup fee?",
        "ground_truth": "A new scooter can be returned within 15 days of delivery, and the pickup fee is 1500 INR "
                        "(not charged if the scooter arrived damaged).",
        "evidence": ["within 15 days of delivery", "pickup fee of 1500 INR"],       # needs TWO chunks
    },
    {
        "id": "q4",
        "question": "When is the first free service due and what does a basic service cost?",
        "ground_truth": "The first free service is due at 500 km or 3 months, whichever comes first, "
                        "and a basic service costs 800 INR.",
        "evidence": ["due at 500 km or 3 months", "basic service costs 800 INR"],    # needs TWO chunks
    },
    {
        "id": "q5",
        "question": "Where are the authorised service centres in Pune?",
        "ground_truth": "The authorised service centres in Pune are in Kothrud, Hinjewadi and Viman Nagar.",
        "evidence": ["Kothrud, Hinjewadi and Viman Nagar"],
    },
    {
        "id": "q6",
        "question": "Do you offer EMI or financing options?",
        "ground_truth": "The documents say nothing about EMI or financing, so this cannot be answered from them.",
        "evidence": [],                                                              # nothing relevant exists
    },
]


# ---------- Metric helpers ----------
def norm(text: str) -> str:
    """Lowercase and collapse whitespace, so matching ignores case and line breaks."""
    return " ".join(text.lower().split())


def relevant_chunk_ids(chunks: list, evidence: list) -> set:
    """Chunks that contain at least one evidence snippet."""
    return {c["chunk_id"] for c in chunks if any(norm(e) in norm(c["text"]) for e in evidence)}


def context_precision(retrieved_ids: list, relevant_ids: set) -> float:
    """Share of the retrieved chunks that are relevant. No relevant chunk exists -> everything is noise -> 0."""
    if not retrieved_ids:
        return 0.0
    return sum(1 for cid in retrieved_ids if cid in relevant_ids) / len(retrieved_ids)


def context_recall(retrieved_texts: list, evidence: list):
    """Share of the evidence snippets found in the retrieved text. None if there is no evidence to find."""
    if not evidence:
        return None
    joined = norm(" ".join(retrieved_texts))
    return sum(1 for e in evidence if norm(e) in joined) / len(evidence)


# ---------- The example from your notes: precision = 40%, recall = 40% ----------
retrieved, hits, total_relevant = 5, 2, 5      # 5 chunks retrieved, 2 relevant; 5 relevant chunks exist
print("Notes example")
print(f"  Context precision = {hits}/{retrieved} = {hits / retrieved:.0%}  -> {1 - hits / retrieved:.0%} of retrieved chunks were useless noise")
print(f"  Context recall    = {hits}/{total_relevant} = {hits / total_relevant:.0%}  -> {1 - hits / total_relevant:.0%} of relevant chunks were missed")


# ---------- Measure OUR retriever on the test set (no LLM needed, so this is free and fast) ----------
def evaluate_retrieval(store_name: str, k: int) -> pd.DataFrame:
    store = STORES[store_name]
    rows = []
    for item in EVAL_SET:
        retrieved = retrieve(item["question"], store, k)
        retrieved_ids = [c["chunk_id"] for c in retrieved]
        relevant = relevant_chunk_ids(store.chunks, item["evidence"])
        rows.append({
            "id": item["id"],
            "retrieved": retrieved_ids,
            "relevant chunks": sorted(relevant),
            "precision": round(context_precision(retrieved_ids, relevant), 2),
            "recall": context_recall([c["text"] for c in retrieved], item["evidence"]),
        })
    return pd.DataFrame(rows)


print("\nStore: paragraph, k = 3")
per_question = evaluate_retrieval("paragraph", k=3)
display(per_question)

# ---------- What happens when we change k? (more chunks: recall can only go up, precision usually drops) ----------
sweep = []
for k in [1, 2, 3, 5]:
    table = evaluate_retrieval("paragraph", k)
    sweep.append({
        "k": k,
        "mean precision": round(table["precision"].mean(), 2),
        "mean recall": round(table["recall"].dropna().astype(float).mean(), 2),   # q6 has no evidence -> skipped
    })
print("\nEffect of k (paragraph store)")
display(pd.DataFrame(sweep))

Notes example
  Context precision = 2/5 = 40%  -> 60% of retrieved chunks were useless noise
  Context recall    = 2/5 = 40%  -> 60% of relevant chunks were missed

Store: paragraph, k = 3


,id,retrieved,relevant chunks,precision,recall
0,q1,"[warranty-0, warranty-1, charging-0]",[warranty-0],0.33,1.0
1,q2,"[charging-0, warranty-0, charging-2]",[charging-0],0.33,1.0
2,q3,"[returns-0, returns-2, returns-1]","[returns-0, returns-2]",0.67,1.0
3,q4,"[service-0, delivery-3, delivery-2]","[service-0, service-1]",0.33,0.5
4,q5,"[service-2, service-0, delivery-2]",[service-2],0.33,1.0
5,q6,"[delivery-3, delivery-2, service-0]",[],0.00,NaN



Effect of k (paragraph store)


,k,mean precision,mean recall
0,1,0.83,0.8
1,2,0.50,0.9
2,3,0.33,0.9
3,5,0.23,1.0


In [ ]:
# ---------- Cell 10: LLM as Judge (comparing the RAG answer with the pre-written correct answer) ----------

# Your notes: we have MANY questions with pre-written correct answers (ground truth).
# To compare the ground truth with the answer from the RAG system we use ANOTHER, BIGGER, more powerful LLM.
# This technique is called "LLM as Judge".
#
# Why not simply compare the two texts? Because the same fact can be worded in many ways:
#   ground truth: "The Zephyr S1 has a 3-year warranty."
#   RAG answer  : "You get three years of warranty coverage on the S1."   <- correct, but the strings differ
# A string comparison calls this wrong. A judge LLM understands that the meaning is the same.

VERDICT_SCORES = {"correct": 1.0, "partially_correct": 0.5, "incorrect": 0.0}

JUDGE_ANSWER_PROMPT = (
    "You are a strict but fair grader. You get a QUESTION, the GROUND TRUTH answer (written by a human expert) "
    "and a CANDIDATE answer from an AI system. Decide whether the candidate gives the same information as the "
    "ground truth. Different wording is fine. Rules:\n"
    "- correct: contains all key facts of the ground truth and nothing that contradicts it.\n"
    "- partially_correct: what it says is right, but it misses at least one key fact.\n"
    "- incorrect: a key fact is wrong, contradicts the ground truth, or is made up.\n"
    "If the ground truth says the question cannot be answered from the documents, an honest 'I don't know' is "
    "correct and any made-up answer is incorrect.\n"
    'Reply ONLY with JSON: {"verdict": "correct" or "partially_correct" or "incorrect", '
    '"reason": "one short sentence"}'
)


def judge_answer(question: str, ground_truth: str, answer: str) -> dict:
    """Ask the big judge LLM to grade one answer against its ground truth."""
    out = ask_json(
        JUDGE_ANSWER_PROMPT,
        f"QUESTION: {question}\n\nGROUND TRUTH: {ground_truth}\n\nCANDIDATE: {answer}",
    )
    verdict = out.get("verdict", "incorrect")
    if verdict not in VERDICT_SCORES:           # an unexpected label counts as incorrect (the safe choice)
        verdict = "incorrect"
    return {"verdict": verdict, "score": VERDICT_SCORES[verdict], "reason": out.get("reason", "")}


def show_judgement(label: str, item: dict, answer: str, judgement: dict) -> None:
    exact_match = answer.strip().lower() == item["ground_truth"].strip().lower()
    print(f"\n=== {label} ===")
    print(f"Q           : {item['question']}")
    print(f"ground truth: {item['ground_truth']}")
    print(f"answer      : {answer}")
    print(f"string match: {exact_match}     |     judge: {judgement['verdict']} ({judgement['score']})")
    print(f"judge reason: {judgement['reason']}")


EVAL_BY_ID = {item["id"]: item for item in EVAL_SET}

# ---------- Hand-written candidate answers, so we KNOW what the right verdict should be ----------
cases = [
    ("Case 1: right fact, different wording", "q1", "You get three years of warranty coverage on the S1.", "correct"),
    ("Case 2: right but incomplete (fee missing)", "q3", "You can return a new scooter within 15 days of delivery.", "partially_correct"),
    ("Case 3: wrong number", "q1", "The Zephyr S1 has a 2-year warranty.", "incorrect"),
    ("Case 4: honest refusal on an unanswerable question", "q6", "I don't know based on the available documents.", "correct"),
    ("Case 5: invented answer on an unanswerable question", "q6", "Yes, we offer no-cost EMI for 12 months on all Zephyr scooters.", "incorrect"),
]
hits = 0
for label, qid, answer, expected in cases:
    item = EVAL_BY_ID[qid]
    judgement = judge_answer(item["question"], item["ground_truth"], answer)
    show_judgement(label, item, answer, judgement)
    hits += judgement["verdict"] == expected
print(f"\nJudge agreed with the expected verdict in {hits} of {len(cases)} hand-written cases.")

# ---------- One REAL answer from our RAG system (q4 needs two facts, so it is a good test) ----------
item = EVAL_BY_ID["q4"]
rag = rag_answer(item["question"], style="strict")
show_judgement("Real RAG answer (q4)", item, rag["answer"], judge_answer(item["question"], item["ground_truth"], rag["answer"]))


=== Case 1: right fact, different wording ===
Q           : How long is the warranty on the Zephyr S1?
ground truth: The Zephyr S1 has a 3-year warranty.
answer      : You get three years of warranty coverage on the S1.
string match: False     |     judge: correct (1.0)
judge reason: Candidate correctly states the 3-year warranty

=== Case 2: right but incomplete (fee missing) ===
Q           : How many days do I have to return a new scooter, and how much is the pickup fee?
ground truth: A new scooter can be returned within 15 days of delivery, and the pickup fee is 1500 INR (not charged if the scooter arrived damaged).
answer      : You can return a new scooter within 15 days of delivery.
string match: False     |     judge: partially_correct (0.5)
judge reason: Omits the pickup fee information

=== Case 3: wrong number ===
Q           : How long is the warranty on the Zephyr S1?
ground truth: The Zephyr S1 has a 3-year warranty.
answer      : The Zephyr S1 has a 2-year warranty.
str